Import data and create dataframe

In [284]:
import pandas as pd

data = pd.read_excel(r"C:\Users\clair\Senior-Project\EPA Data Cleaning\Raw Data\Excel\ExtraEPAData_diesel.xlsx")

In [285]:
df = pd.DataFrame(data)
df.head()

,Unnamed: 0,(R) 2000,(R) 2001,(R) 2002,(R) 2003,(R) 2004,(R) 2005,(R) 2006,(R) 2007,(R) 2008,...,(R) 2021,(P) 2022,(P) 2023,(P) 2024,(P) 2025,(P) 2026,(P) 2027,(P) 2028,(P) 2029,(P) 2030
0,Light-duty vehicles,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Total HCa,2.625,2.602,2.501,2.435,2.374,2.288,2.148,2.103,2.064,...,0.393,0.437,0.297,0.250,0.242,0.188,0.195,0.172,0.173,0.173
2,Exhaust CO,45.904,44.636,41.905,40.430,39.174,37.434,35.151,34.404,33.744,...,8.159,9.039,7.490,6.935,6.951,6.227,6.477,6.436,6.745,7.043
3,Exhaust NOx,3.052,2.862,2.549,2.488,2.432,2.342,2.210,2.175,2.135,...,0.364,0.389,0.354,0.276,0.247,0.175,0.184,0.145,0.146,0.147
4,Exhaust PM2.5,0.061,0.064,0.068,0.070,0.070,0.069,0.065,0.062,0.061,...,0.009,0.010,0.008,0.006,0.005,0.004,0.004,0.004,0.004,0.004


Remove (R) and (P) from years

In [286]:
df.columns = df.columns.str.replace(r"\(R\)\s*", "", regex=True)
df.columns = df.columns.str.replace(r"\(P\)\s*", "", regex=True)
df.head()

,Unnamed: 0,2000,2001,2002,2003,2004,2005,2006,2007,2008,...,2021,2022,2023,2024,2025,2026,2027,2028,2029,2030
0,Light-duty vehicles,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Total HCa,2.625,2.602,2.501,2.435,2.374,2.288,2.148,2.103,2.064,...,0.393,0.437,0.297,0.250,0.242,0.188,0.195,0.172,0.173,0.173
2,Exhaust CO,45.904,44.636,41.905,40.430,39.174,37.434,35.151,34.404,33.744,...,8.159,9.039,7.490,6.935,6.951,6.227,6.477,6.436,6.745,7.043
3,Exhaust NOx,3.052,2.862,2.549,2.488,2.432,2.342,2.210,2.175,2.135,...,0.364,0.389,0.354,0.276,0.247,0.175,0.184,0.145,0.146,0.147
4,Exhaust PM2.5,0.061,0.064,0.068,0.070,0.070,0.069,0.065,0.062,0.061,...,0.009,0.010,0.008,0.006,0.005,0.004,0.004,0.004,0.004,0.004


Rename unnamed column to Label

In [287]:
df = df.rename(columns={"Unnamed: 0": "Label"})
df.head()

,Label,2000,2001,2002,2003,2004,2005,2006,2007,2008,...,2021,2022,2023,2024,2025,2026,2027,2028,2029,2030
0,Light-duty vehicles,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Total HCa,2.625,2.602,2.501,2.435,2.374,2.288,2.148,2.103,2.064,...,0.393,0.437,0.297,0.250,0.242,0.188,0.195,0.172,0.173,0.173
2,Exhaust CO,45.904,44.636,41.905,40.430,39.174,37.434,35.151,34.404,33.744,...,8.159,9.039,7.490,6.935,6.951,6.227,6.477,6.436,6.745,7.043
3,Exhaust NOx,3.052,2.862,2.549,2.488,2.432,2.342,2.210,2.175,2.135,...,0.364,0.389,0.354,0.276,0.247,0.175,0.184,0.145,0.146,0.147
4,Exhaust PM2.5,0.061,0.064,0.068,0.070,0.070,0.069,0.065,0.062,0.061,...,0.009,0.010,0.008,0.006,0.005,0.004,0.004,0.004,0.004,0.004


Transpose df, make years rows instead of columns

In [288]:
vehicle_mask = df[df.columns[1:]].isna().all(axis=1)

In [289]:
df["VehicleType"] = df["Label"].where(vehicle_mask)
df["VehicleType"] = df["VehicleType"].ffill()

In [290]:
df = df[~vehicle_mask | df["Label"].isin(df["VehicleType"])]
df = df[df["Label"] != "GASOLINE"]

df = df.dropna(subset=df.columns[1:-1], how="all")

df["Measure"] = df["VehicleType"] + " - " + df["Label"]

df_t = (
    df
    .set_index("Measure")
    .drop(columns=["Label", "VehicleType"])
    .T
)

df_t.index.name = "Year"

In [291]:
df_t.head()

Measure,Light-duty vehicles - Total HCa,Light-duty vehicles - Exhaust CO,Light-duty vehicles - Exhaust NOx,Light-duty vehicles - Exhaust PM2.5,Light-duty vehicles - Brake Wear PM2.5,Light-duty vehicles - Tire Wear PM2.5,Light-duty vehicles - Exhaust CO2,Light-duty vehicles - Energy Consumption (Btu/mile),Light-duty trucks - Total HCa,Light-duty trucks - Exhaust CO,...,Buses - Exhaust CO2,Buses - Energy Consumption (Btu/mile),Heavy-duty vehicles (other than buses) - Total HCa,Heavy-duty vehicles (other than buses) - Exhaust CO,Heavy-duty vehicles (other than buses) - Exhaust NOx,Heavy-duty vehicles (other than buses) - Exhaust PM2.5,Heavy-duty vehicles (other than buses) - Brake Wear PM2.5,Heavy-duty vehicles (other than buses) - Tire Wear PM2.5,Heavy-duty vehicles (other than buses) - Exhaust CO2,Heavy-duty vehicles (other than buses) - Energy Consumption (Btu/mile)
Year,,,,,,,,,,,,,,,,,,,,,
2000,2.625,45.904,3.052,0.061,0.003,0.001,314.446,4019.922,1.609,22.842,...,1557.343,19909.315,0.937,4.633,25.035,1.052,0.012,0.004,1676.602,21433.921
2001,2.602,44.636,2.862,0.064,0.003,0.001,317.077,4053.565,1.509,20.744,...,1555.379,19884.216,0.934,4.627,24.060,1.002,0.012,0.004,1677.044,21439.580
2002,2.501,41.905,2.549,0.068,0.003,0.001,320.800,4101.158,1.351,18.013,...,1552.564,19848.200,0.938,4.652,23.083,0.950,0.012,0.004,1669.223,21339.601
2003,2.435,40.430,2.488,0.070,0.003,0.001,325.045,4155.418,1.295,16.042,...,1545.783,19761.528,0.955,4.717,21.635,0.911,0.012,0.004,1664.819,21283.302
2004,2.374,39.174,2.432,0.070,0.003,0.001,328.818,4203.655,1.236,14.504,...,1547.839,19787.805,0.987,4.860,20.342,0.886,0.011,0.004,1664.828,21283.420


Filter to only CO2 emissions columns

In [292]:
co2_df = df_t.filter(like="CO2")
co2_df.head()

Measure,Light-duty vehicles - Exhaust CO2,Light-duty trucks - Exhaust CO2,Buses - Exhaust CO2,Heavy-duty vehicles (other than buses) - Exhaust CO2
Year,,,,
2000,314.446,616.964,1557.343,1676.602
2001,317.077,644.687,1555.379,1677.044
2002,320.800,668.929,1552.564,1669.223
2003,325.045,693.706,1545.783,1664.819
2004,328.818,715.243,1547.839,1664.828


Average CO2 emissions across years for vehicle types

In [293]:
df_clean = co2_df.mean().reset_index()
df_clean.columns = ['Vehicle Type', 'Average_CO2']

df_clean.head()

,Vehicle Type,Average_CO2
0,Light-duty vehicles - Exhaust CO2,366.391484
1,Light-duty trucks - Exhaust CO2,669.968548
2,Buses - Exhaust CO2,1537.465290
3,Heavy-duty vehicles (other than buses) - Exha...,1554.169000


In [294]:
#df_clean = averages[averages['Vehicle Type'] != 'Both'].copy()
df_clean['Vehicle Type'] = df_clean['Vehicle Type'].str.replace(' - Exhaust CO2', '', regex=False)

df_clean['Vehicle Type'] = df_clean['Vehicle Type'].str.replace('Motorcyclesb', 'Motorcycles', regex=False)

df_clean.head()

,Vehicle Type,Average_CO2
0,Light-duty vehicles,366.391484
1,Light-duty trucks,669.968548
2,Buses,1537.465290
3,Heavy-duty vehicles (other than buses),1554.169000


In [295]:
#df_clean.to_csv('clean_gasoline.csv')
df_clean.to_csv('clean_diesel.csv')